# Returns Forecasting: Training Pipeline

This notebook trains two forecasting models for portfolio optimization:

1. **LightGBM** — predicts expected returns per asset (replaces historical mean)
2. **GARCH** — forecasts the covariance matrix (replaces historical covariance)

Both models are exported for deployment to CAII (Cloudera AI Inference Services).

## 1. Setup and Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portfolio_optimization.utils import download_data, get_input_data, calculate_returns
from portfolio_optimization.settings import ReturnsComputeSettings
from portfolio_optimization.forecasting.config import ForecastingConfig
from portfolio_optimization.forecasting.feature_engineering import (
    compute_features, build_training_data, flatten_features_for_training
)
from portfolio_optimization.forecasting.lightgbm_model import ReturnsForecaster
from portfolio_optimization.forecasting.garch_model import CovarianceForecaster
from portfolio_optimization.forecasting.caii_client import ForecastClient

print("Imports OK")

Imports OK


In [2]:
# Download DOW30 data if not already present
data_dir = "../data/stock_data"
download_data(data_dir, datasets=["dow30"])

prices = get_input_data(f"{data_dir}/dow30.csv")
print(f"Price data: {prices.shape[0]} days x {prices.shape[1]} assets")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
prices.tail()

Saved 30 tickers to ../data/stock_data/dow30.csv
Price data: 5033 days x 30 assets
Date range: 2005-01-03 to 2024-12-31


,AAPL,AMGN,AMZN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,...,MRK,MSFT,NKE,NVDA,PG,SHW,TRV,UNH,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-24,256.339783,251.015900,229.050003,297.801178,179.339996,359.902527,340.463531,57.506935,133.733002,110.482224,...,94.462715,433.363556,73.096382,140.010895,161.007751,341.028076,237.124268,487.342163,35.454536,91.220375
2024-12-26,257.153809,249.772644,227.050003,298.321198,180.380005,359.461945,337.784821,57.631844,133.863190,110.472420,...,94.861656,432.160095,73.239166,139.721329,162.170456,340.034149,237.954102,492.204987,35.597069,91.328636
2024-12-27,253.748535,249.269638,223.750000,295.436035,180.720001,357.249023,334.552521,57.276329,133.881790,109.490883,...,94.700180,424.683014,72.744171,136.805649,161.570038,337.849335,235.689102,491.087982,35.561440,90.216446
2024-12-30,250.382950,246.090271,221.300003,291.785461,176.550003,355.437622,331.873627,56.872772,133.017151,108.754723,...,93.436890,419.060455,71.059319,137.284958,159.244614,333.381226,234.185577,488.979156,35.276379,89.143600
2024-12-31,248.615784,247.362045,219.389999,291.255524,177.000000,355.192841,330.479858,56.882381,134.662735,109.294563,...,94.491219,415.775665,72.030251,134.089722,159.778290,334.542542,235.181427,487.111053,35.623798,88.927063


## 2. Configuration

Key parameters:
- `forecast_horizon`: days ahead to predict (5 = weekly)
- `training_window`: lookback days for training (~756 = 3 years)
- Feature windows for momentum, volatility, RSI

In [3]:
config = ForecastingConfig(
    forecast_horizon=5,
    training_window=756,
    return_type="LOG",
)
print(config.model_dump_json(indent=2))

{
  "forecast_horizon": 5,
  "training_window": 756,
  "retrain_frequency": 21,
  "return_type": "LOG",
  "features": {
    "momentum_windows": [
      5,
      10,
      21
    ],
    "volatility_windows": [
      10,
      21
    ],
    "rsi_window": 14,
    "include_cross_asset_corr": true,
    "cross_asset_corr_window": 21
  },
  "lightgbm": {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 20,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "early_stopping_rounds": 50,
    "seed": 42
  },
  "garch": {
    "p": 1,
    "q": 1,
    "vol_model": "GARCH",
    "mean_model": "Constant",
    "dist": "t"
  },
  "caii_endpoint": null,
  "caii_covariance_endpoint": null
}


## 3. Feature Engineering

Per-asset features: momentum, rolling volatility, RSI, price vs SMA, cross-asset correlation.

In [4]:
features = compute_features(prices, config.features)
print(f"Feature matrix: {features.shape}")
print(f"\nFeatures per asset:")
sample_ticker = prices.columns[0]
print([col for col in features[sample_ticker].columns])

Feature matrix: (4970, 330)

Features per asset:
['momentum_5d', 'momentum_10d', 'momentum_21d', 'volatility_10d', 'volatility_21d', 'rsi_14d', 'mean_return_21d', 'price_vs_sma21', 'price_vs_sma63', 'return_dispersion_10d', 'mean_cross_corr']


In [5]:
X, y = build_training_data(
    prices,
    forecast_horizon=config.forecast_horizon,
    return_type=config.return_type,
    config=config.features,
)
X_flat, y_flat = flatten_features_for_training(X, y)
print(f"Flattened training set: {X_flat.shape[0]} samples x {X_flat.shape[1]} features")
print(f"Target distribution: mean={y_flat.mean():.6f}, std={y_flat.std():.4f}")

Flattened training set: 148950 samples x 12 features
Target distribution: mean=0.002466, std=0.0394


## 4. Train LightGBM (Expected Returns)

In [6]:
returns_forecaster = ReturnsForecaster(config)
metrics = returns_forecaster.train(prices)
print(f"\nTraining metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

ModuleNotFoundError: No module named 'lightgbm'

In [7]:
importance = returns_forecaster.feature_importance()
print("Top 10 features by importance:")
print(importance.head(10).to_string(index=False))

RuntimeError: Model not trained.

In [ ]:
# Predict expected returns using the most recent data
predicted_returns = returns_forecaster.predict(prices)
print("Predicted expected returns (next 5 days):")
for ticker, ret in zip(prices.columns, predicted_returns):
    print(f"  {ticker}: {ret:+.6f}")

## 5. Train GARCH (Covariance Forecasting)

In [ ]:
cov_forecaster = CovarianceForecaster(config)
garch_metrics = cov_forecaster.train(prices)

converged = sum(1 for m in garch_metrics.values() if m.get("converged"))
print(f"GARCH models converged: {converged}/{len(garch_metrics)}")
print(f"\nSample metrics (first 5 assets):")
for ticker in list(garch_metrics.keys())[:5]:
    m = garch_metrics[ticker]
    if m.get("converged"):
        print(f"  {ticker}: AIC={m['aic']:.1f}, persistence={m['persistence']:.4f}")
    else:
        print(f"  {ticker}: FAILED - {m.get('error', 'unknown')}")

In [ ]:
# Forecast covariance matrix
predicted_cov = cov_forecaster.predict(prices)
predicted_vol = np.sqrt(np.diag(predicted_cov))
print("Predicted annualized volatility (top 5):")
vol_series = pd.Series(predicted_vol * np.sqrt(252), index=prices.columns)
print(vol_series.sort_values(ascending=False).head().to_string())

## 6. Compare: Historical vs. Forecasted Inputs

In [ ]:
# Historical baseline
returns_dict = calculate_returns(
    prices,
    returns_compute_settings=ReturnsComputeSettings(return_type="LOG", freq=1),
)

hist_mean = returns_dict["mean"]
hist_cov = returns_dict["covariance"]

print("Mean return comparison (annualized):")
comparison = pd.DataFrame({
    "Historical": hist_mean * 252,
    "LightGBM Forecast": predicted_returns * 252 / config.forecast_horizon,
}, index=prices.columns)
comparison["Difference"] = comparison["LightGBM Forecast"] - comparison["Historical"]
print(comparison.sort_values("Difference", ascending=False).head(10).to_string())

## 7. Export Models

- LightGBM → native model file
- GARCH → JSON parameters (GARCH is recursive, no serialized-model format)

In [ ]:
import os
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

# Save LightGBM (native format)
lgb_path = returns_forecaster.save(f"{model_dir}/returns_forecaster.lgb")
print(f"LightGBM saved: {lgb_path}")

# Export GARCH parameters
garch_path = cov_forecaster.export_params(f"{model_dir}/garch_params.json")
print(f"GARCH params saved: {garch_path}")

## 7b. Register in MLflow (for CAII Deployment)

MLflow model registry is the bridge to CAII — register the native LightGBM model so CAII can deploy it as an inference endpoint.

In [ ]:
# Register LightGBM returns forecaster in MLflow → CAII can deploy this
returns_uri = returns_forecaster.register_mlflow(
    model_name="PortfolioReturnsForecaster"
)
print(f"Returns model URI: {returns_uri}")

# Register GARCH covariance forecaster params in MLflow
cov_uri = cov_forecaster.register_mlflow(
    model_name="PortfolioCovarianceForecaster"
)
print(f"Covariance model URI: {cov_uri}")

## 8. Integration with Optimizer

Use `ForecastClient.update_returns_dict()` to swap in forecasted values before optimization.

In [ ]:
# Create forecast client with local models
client = ForecastClient(
    config=config,
    returns_model=returns_forecaster,
    covariance_model=cov_forecaster,
)

# Get returns_dict with historical values, then replace with forecasts
returns_dict_forecasted = client.update_returns_dict(returns_dict, prices)

print("returns_dict updated with forecasted values.")
print(f"  mean shape: {returns_dict_forecasted['mean'].shape}")
print(f"  covariance shape: {returns_dict_forecasted['covariance'].shape}")
print("\nReady for optimizer: pass returns_dict_forecasted to CVaR or MeanVariance.")